# 抽样与置信区间

本 Notebook 对应 `TASK.md`。差异一律定义为：实验组转化率减去对照组转化率。

In [1]:
import math
import numpy as np
import pandas as pd

control_success, control_n = 100, 1000
treatment_success, treatment_n = 120, 1000

## 正态近似

完成函数，返回 `diff, se, lower, upper`。

In [4]:
def normal_ci(control_success, control_n, treatment_success, treatment_n):
    # TODO：计算两组转化率、实验组减对照组的差异、标准误与95%区间。

    # 1.算各自的转化率
    p_c = control_success / control_n
    p_t = treatment_success / treatment_n

    # 2.算差异
    diff = p_c - p_t

    # 3.算标准误
    se = math.sqrt((p_c * (1 - p_c) / control_n + (p_t * (1 - p_t) / treatment_n)))

    # 4.算区间(系数取1.96)
    lower = diff - 1.96 * se
    upper = diff + 1.96 * se

    return diff, se, lower, upper

normal_ci(control_success, control_n, treatment_success, treatment_n)

# diff, se, lower, upper = normal_ci(control_success, control_n, treatment_success, treatment_n)
# print(diff, se, lower, upper)

(-0.01999999999999999,
 0.013985706989637671,
 -0.047411985699689824,
 0.007411985699689844)

## 模拟用户级数据

构造每组1000位用户的 `group`、`converted` 数据。先让对照组有100个1，实验组有120个1；随后可打乱行顺序。

In [ ]:
# TODO：构造 data，列名必须为 group 与 converted。
df_control = pd.DataFrame({
    'group': "control",
    'converted': [1] * 100 + [0] * 900
})

df_treatment = pd.DataFrame({
    'group': "treatment",
    'converted': [1] * 120 + [0] * 880
})
data = pd.concat([df_control, df_treatment], ignore_index=True)
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
display(data.head())
print(data.groupby('group')['converted'].agg(['count', 'sum', 'mean'])) # .agg：聚合（Aggregate），一次性计算多个指标，相当于.mean().sum()等

,group,converted
0,treatment,0
1,control,0
2,treatment,0
3,control,0
4,treatment,0


           count  sum  mean
group                      
control     1000  100  0.10
treatment   1000  120  0.12


## Bootstrap

在每个组内分别有放回重抽样。每次计算实验组均值减对照组均值；最后返回2.5%与97.5%分位数。

In [ ]:
def bootstrap_ci(data, repeats=3000, seed=42):
    # TODO：不要把两组混合后抽样。
    np.random.seed(seed)
    # 筛选出对照组，只取 converted 这一列，转成 numpy 数组
    ctrl_vals = data[data['group'] == 'control']['converted'].to_numpy()   # [1, 0, 1......]
    treat_vals = data[data['group'] == 'treatment']['converted'].to_numpy()

    boot_diffs = []

    for _ in range(repeats):
        # 从 ctrl_vals 里随机挑出 len(ctrl_cals) 个数，replace=True 表示抽完放回
        sample_ctrl = np.random.choice(ctrl_vals,size=len(ctrl_vals),replace=True)  # [0, 0, 1......]
        sample_treat = np.random.choice(treat_vals,size=len(treat_vals),replace=True)

        # 计算差异并存入列表
        diff = sample_treat.mean() - sample_ctrl.mean() # 1的个数 / 1000
        boot_diffs.append(diff)

    # 3.取 2.5% 与 97.5% 分位数
    lower = float(np.percentile(boot_diffs, 2.5))
    upper = float(np.percentile(boot_diffs, 97.5))

    return lower, upper

lower_boot, upper_boot = bootstrap_ci(data)
print(lower_boot, upper_boot)   # 0.8%， 4.7%
# 根据当前实验数据，新方案相比于旧方案带来的真实转化率变化，合理波动的范围在 -0.8 个百分点到 +4.7 个百分点之间。
# 这两个数字代表的是两组转化率的实际差值（实验组转化率 - 对照组转化率）

-0.008024999999999997 0.047000000000000014


## 写给业务方

在下面写一句不包含“真实差异有95%概率在区间内”的解释。说明当区间包含0时，当前数据意味着什么。

**你的解释：**

填写这里。

## 完成后

将两个函数复制到同目录 `solution.py`，再运行 `python run_tests.py`。当前本机 pandas/numpy 环境不兼容时，先按 `../README.md` 的环境说明修复。